In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [6]:
MAX_TOKENS = 150

In [8]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = llm.invoke(messages)

    return {"messages": [response]}

In [9]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [10]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [20]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Nitish."}]},
    config,
)

result["messages"][-1].text

Current Token Count -> 10
Hi, my name is Nitish.


"Hi Nitish! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?"

In [21]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].text

Current Token Count -> 51
Hi, my name is Nitish.
[{'type': 'text', 'text': "Hi Nitish! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EnMKcQFpFH0T5jS+zHQiOlKyWQFwAbWyxJNTTiLaD6MT6N+qvYQp5AjojQUyWFRz1fhqTZJ2RruNdmMmVKYMpSrPJ1a9bdL+vvFpI+GTho0TiUVpdJpHFfSzvuo4V21Fyx4NX7a0zkXXcHpH22wSe1jsw+BQ'}}]
I am learning LangGraph.


'That’s a fantastic choice! **LangGraph** is currently one of the most powerful ways to build agentic workflows because it allows you to create cyclic graphs (loops), which are essential for agents that need to "think," "act," and "correct" themselves.\n\nSince you are just starting out, are you looking for a high-level overview of the concepts, or are you ready to dive into some code?\n\nTo help orient you, here are the **four core concepts** you should wrap your head around first:\n\n1.  **State:** This is the "memory" of your graph. It’s a shared data structure (usually a TypedDict) that gets passed between nodes. Every node reads and updates this state.\n2.  **Nodes:** These are your Python functions. They represent the "units of work" (e.g., calling an LLM, searching the web, or calculating a value).\n3.  **Edges:** These define the control flow. They tell the graph what to do after a node finishes. You can have simple edges (Node A -> Node B) or **Conditional Edges** (if X, go to

In [22]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].text

Current Token Count -> 13
Can you explain short term memory?


'Short-term memory (STM), often referred to as **"working memory,"** is the brain\'s system for temporarily holding a small amount of information in an active, readily available state.\n\nThink of it as your brain’s **"scratchpad" or "RAM"**: it’s where you keep information you need *right now* before you either use it and discard it, or move it into long-term storage.\n\nHere is a breakdown of how it works:\n\n---\n\n### 1. The Capacity: The "7 ± 2" Rule\nIn the 1950s, psychologist George Miller famously suggested that the average person can hold about **seven items** (plus or minus two) in their short-term memory at once. \n\n*   **Example:** If someone reads you a 10-digit phone number, you might remember the first few and the last few, but lose the middle ones unless you group them (a process called **chunking**). By grouping the digits into three sets (e.g., 555-012-3456), you turn 10 individual numbers into three "chunks," making them easier to hold.\n\n### 2. The Duration: "Use 

In [23]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.text)
    print('-'*120)

Hi, my name is Nitish.
------------------------------------------------------------------------------------------------------------------------
Hi Nitish! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?
------------------------------------------------------------------------------------------------------------------------
Hi, my name is Nitish.
------------------------------------------------------------------------------------------------------------------------
Hi again, Nitish! It’s nice to see you. How can I help you out today?
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
That’s fantastic! LangGraph is a powerful tool—it’s essentially the "next step" for anyone moving from simple LLM chains to complex, agentic workflows